# Практика 12. Довірчі інтервали для середнього і частки

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса  
**Липнева середньомісячна температура:** 23 °C

Мета: на вибірці денних температур побудувати довірчі інтервали для середнього через t-розподіл і для частки через нормальне наближення, порівняти інтервали різного рівня довіри та правильно інтерпретувати їх.

## Завдання 1. Вибірка денних температур

Генеруємо 30 денних температур навколо липневого середнього Одеси. `ddof=1` обчислює вибіркове стандартне відхилення, яке використовуватимемо для оцінки невідомого σ.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

variant = 3
july_mean = 23
np.random.seed(variant)
daily_temps = np.random.normal(loc=july_mean, scale=2.5, size=30)

print("Денні температури (°C):")
print(daily_temps)
print(f"Вибіркове середнє: {daily_temps.mean():.3f} °C")
print(f"Вибіркове стандартне відхилення: {daily_temps.std(ddof=1):.3f} °C")

## Завдання 2. Довірчий інтервал для середньої температури

Оскільки справжнє стандартне відхилення температур генеральної сукупності σ невідоме й оцінюється за цією невеликою вибіркою, використовуємо t-розподіл із `n - 1` ступенями вільності.

In [ ]:
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)
n = len(daily_temps)
se = s / np.sqrt(n)
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print(f"n = {n}, x̄ = {mean:.3f} °C, s = {s:.3f} °C, SE = {se:.3f} °C")
print(f"95% довірчий інтервал для середньої температури: ({ci_mean_95[0]:.3f}, {ci_mean_95[1]:.3f}) °C")

Тут не можна просто замінити t-критичне значення z-критичним: справжнє σ генеральної сукупності невідоме, а вибірка невелика (`n=30`). Ми оцінюємо σ за вибірковим `s`, тому враховуємо додаткову невизначеність за допомогою t-розподілу з `n-1` ступенями вільності.

## Завдання 3. Довірчий інтервал для частки

Оцінимо частку днів із температурою вище табличного липневого середнього 23 °C. Для нормального наближення перевіримо, чи достатньо великими є очікувані кількості успіхів і неуспіхів.

In [ ]:
above_july_mean = daily_temps > july_mean
p_hat = above_july_mean.mean()
n_p = n * p_hat
n_1p = n * (1 - p_hat)
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)
contains_half = ci_prop_95[0] <= 0.5 <= ci_prop_95[1]

print(f"Кількість днів вище {july_mean} °C: {above_july_mean.sum()} із {n}")
print(f"Оцінка частки p̂: {p_hat:.3f}")
print(f"n·p̂ = {n_p:.2f}; n·(1-p̂) = {n_1p:.2f}")
print(f"95% довірчий інтервал для частки: ({ci_prop_95[0]:.3f}, {ci_prop_95[1]:.3f})")
print(f"Чи потрапляє 0.5 в інтервал? {contains_half}")

Умова звичного нормального наближення для частки вимагає, щоб `n·p̂` і `n·(1-p̂)` були достатньо великими (часто використовують поріг 5). Перевірте виведені значення: для цієї симетричної вибірки обидва мають бути не меншими за 5. За моделлю, симетричною навколо 23 °C, теоретична частка вище 23 °C становить близько 0.5. Результат перевірки `contains_half` показує, чи накрив цей конкретний інтервал 0.5. Якщо ні, це можливо через випадкову мінливість вибірки: процедура 95% довіри не гарантує покриття в кожному окремому випадку, і приблизно 5% інтервалів при повторенні процедури не накривають істинну частку.

## Завдання 4. Ширина інтервалу при різних рівнях довіри

На тих самих даних побудуємо t-інтервали для середнього з рівнями довіри 90%, 95% і 99% та порівняємо їхню ширину.

In [ ]:
confidence_levels = [0.90, 0.95, 0.99]
interval_rows = []
for confidence in confidence_levels:
    interval = stats.t.interval(confidence, df=n - 1, loc=mean, scale=se)
    interval_rows.append({
        "Рівень довіри": confidence,
        "Нижня межа, °C": interval[0],
        "Верхня межа, °C": interval[1],
        "Ширина, °C": interval[1] - interval[0],
    })

intervals_table = pd.DataFrame(interval_rows)
display(intervals_table)

За незмінних даних вищий рівень довіри потребує ширшого інтервалу: щоб частіше накривати невідоме середнє, доводиться охоплювати більший діапазон можливих значень. Щоб звузити 99% інтервал, не знижуючи рівень довіри, потрібно зменшити стандартну похибку — наприклад, зібрати більшу вибірку; додаткові дані потребують часу й ресурсів.

## Завдання 5. Інтерпретація 95% довірчого інтервалу

Побудований 95% інтервал — це результат процедури, застосованої до цієї вибірки. Уявімо, що ми багато разів генеруємо нову вибірку з тієї самої генеральної сукупності й щоразу будуємо інтервал тим самим методом: приблизно 95% таких інтервалів накривали б істинне середнє. Для вже отриманого інтервалу й істинного середнього значення параметр або лежить усередині, або ні; 95% описує довгострокову частку успішного покриття процедури, а не ймовірність для цього конкретного інтервалу.

## Контрольні питання

1. **Точкова оцінка й довірчий інтервал.** Точкова оцінка — одне число, наприклад вибіркове середнє; вона не показує, наскільки оцінка невизначена через скінченний обсяг вибірки. Довірчий інтервал дає діапазон правдоподібних значень параметра з урахуванням варіації вибірки та заданого рівня довіри.

2. **Чому t-розподіл замість нормального?** Коли σ невідоме й оцінюється за вибіркою, а `n` не дуже велике, оцінка середнього має додаткову невизначеність. t-розподіл із `n-1` ступенями вільності враховує її та має важчі хвости, ніж стандартний нормальний розподіл.

3. **Що означає 95% довіри?** Якщо багато разів брати вибірки тим самим способом і будувати для кожної інтервал за тією самою процедурою, близько 95% отриманих інтервалів накриватимуть істинний параметр. Не слід казати, що після спостереження саме цього інтервалу ймовірність потрапляння в нього істинного фіксованого параметра дорівнює 95%.

4. **Що визначає ширину інтервалу?** (а) Рівень довіри: вищий рівень вимагає ширшого інтервалу. (б) Варіативність даних, оцінена через `s`: більше стандартне відхилення збільшує стандартну похибку й ширину. (в) Обсяг вибірки `n`: більше спостережень зменшує стандартну похибку `s/√n` і звужує інтервал.

## Підсумок

Для вибірки денних температур Одеси побудовано t-довірчий інтервал для середнього та нормальний довірчий інтервал для частки з перевіркою умов наближення. Порівняно ширину інтервалів для середнього при рівнях довіри 90%, 95% і 99% та сформульовано частотну інтерпретацію довіри.